# Supervised Peer-to-Peer Generator-Critic Debate Loop

This notebook implements a tri-agent architecture:
- **Agent A (Generator)**: Google AI Studio / Gemini
- **Agent B (Critic)**: OpenRouter Free Tier Model
- **Agent C (Antigravity Supervisor)**: Intercepts intermediate turns to enforce policy, check for hallucinated critiques, and prevent premature consensus.

In [ ]:
import os
import json
from typing import Dict, List, Tuple, Any, Callable, Optional
from openai import OpenAI

# Verify environment variables
gemini_api_key = os.environ.get("GEMINI_API_KEY", os.environ.get("GOOGLE_API_KEY", ""))
openrouter_api_key = os.environ.get("OPENROUTER_API_KEY", "")

env_status = {
    "gemini_key_present": bool(gemini_api_key),
    "openrouter_key_present": bool(openrouter_api_key)
}
print(json.dumps(env_status))

In [ ]:
# Define system prompts for asymmetric roles including Supervisor

GENERATOR_SYSTEM = """You are an expert scientific and technical research assistant.
Your objective is to generate detailed, rigorously reasoned solutions to the user's research query.
When presented with a critique, analyze each point objectively and revise your draft to address every identified vulnerability or gap.
Maintain strict factual accuracy and structured analytical output."""

CRITIC_SYSTEM = """You are an adversarial code and scientific reviewer.
Your sole job is to identify logical flaws, unstated assumptions, missing edge cases, security vulnerabilities, or methodological weaknesses in the draft.
Do NOT write the replacement solution or complete revised text yourself—only provide concise, prioritized critique points.
If and ONLY IF the draft contains zero logical flaws, missing requirements, or methodological gaps, respond strictly with the exact single token: [APPROVED]."""

SUPERVISOR_SYSTEM = """You are the Antigravity Supervisor monitoring a debate between Generator and Critic.
Your job is to audit both Generator draft and Critic review for correctness.
If Critic issued [APPROVED] prematurely while gaps remain, override approval and state missing requirements.
If Critic provided invalid critique, rectify it before passing to Generator."""

print(json.dumps({
    "generator_prompt_len": len(GENERATOR_SYSTEM),
    "critic_prompt_len": len(CRITIC_SYSTEM),
    "supervisor_prompt_len": len(SUPERVISOR_SYSTEM)
}))

In [ ]:
class SupervisedDebateOrchestrator:
    def __init__(self, 
                 gemini_key: str, 
                 openrouter_key: str, 
                 generator_model: str = "gemini-2.5-pro", 
                 critic_model: str = "meta-llama/llama-3.3-70b-instruct:free",
                 supervisor_callback: Optional[Callable[[str, str], Tuple[bool, str]]] = None):
        
        self.gen_client = OpenAI(
            api_key=gemini_key if gemini_key else "dummy_key",
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
        )
        self.critic_client = OpenAI(
            api_key=openrouter_key if openrouter_key else "dummy_key",
            base_url="https://openrouter.ai/api/v1"
        )
        self.generator_model = generator_model
        self.critic_model = critic_model
        self.supervisor_callback = supervisor_callback

    def run_generator_step(self, conversation_history: List[Dict[str, str]]) -> str:
        messages = [{"role": "system", "content": GENERATOR_SYSTEM}] + conversation_history
        response = self.gen_client.chat.completions.create(
            model=self.generator_model,
            messages=messages,
            temperature=0.2
        )
        return response.choices[0].message.content

    def run_critic_step(self, conversation_history: List[Dict[str, str]]) -> str:
        messages = [{"role": "system", "content": CRITIC_SYSTEM}] + conversation_history
        response = self.critic_client.chat.completions.create(
            model=self.critic_model,
            messages=messages,
            temperature=0.1
        )
        return response.choices[0].message.content

    def execute_supervised_debate(self, task_prompt: str, max_rounds: int = 3) -> Dict[str, Any]:
        conversation_history: List[Dict[str, str]] = [
            {"role": "user", "content": f"Task Specification: {task_prompt}"}
        ]
        
        trace = []
        current_draft = ""
        final_approval = False
        
        for round_idx in range(max_rounds):
            # Generator Step
            current_draft = self.run_generator_step(conversation_history)
            conversation_history.append({"role": "assistant", "content": current_draft})
            
            # Critic Step
            raw_critique = self.run_critic_step(conversation_history)
            critic_approved = "[APPROVED]" in raw_critique.strip()
            
            # Antigravity Supervisor Interception
            if self.supervisor_callback:
                supervised_approved, final_critique = self.supervisor_callback(current_draft, raw_critique)
            else:
                supervised_approved, final_critique = critic_approved, raw_critique
                
            trace.append({
                "round": round_idx + 1,
                "draft_length": len(current_draft),
                "critic_approved": critic_approved,
                "supervisor_approved": supervised_approved,
                "critique_length": len(final_critique)
            })
            
            if supervised_approved:
                final_approval = True
                break
                
            conversation_history.append({"role": "user", "content": f"Supervisor-Verified Critique for Revision:\n{final_critique}"})
            
        return {
            "final_draft": current_draft,
            "approved": final_approval,
            "total_rounds": len(trace),
            "trace": trace
        }

print(json.dumps({"supervised_class_initialized": True}))

In [ ]:
# Verification execution block
if __name__ == "__main__":
    def mock_supervisor(draft: str, critique: str) -> Tuple[bool, str]:
        # Supervisor logic check: override premature approval if draft length is too short
        if "[APPROVED]" in critique and len(draft) < 50:
            return False, "Supervisor Override: Draft is under-specified. Provide complete technical detail."
        return "[APPROVED]" in critique, critique

    orchestrator = SupervisedDebateOrchestrator(
        gemini_key=gemini_api_key, 
        openrouter_key=openrouter_api_key,
        supervisor_callback=mock_supervisor
    )
    state_check = {
        "orchestrator_valid": orchestrator is not None,
        "has_supervisor": orchestrator.supervisor_callback is not None
    }
    print(json.dumps(state_check))